# Two-Tower Model (TTN)

Build a PyTorch two-tower model on the Home & Kitchen interactions and the
extracted item features. See `README.md` in this folder for the design.

BPR-style pairwise ranking loss: `-log σ(score(u, i) - score(u, j))`.

In [57]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Load data

Two sources, both keyed on `asin`:

- **Item features** — `data/df_features.pkl`: one row per item (~1.13M items)
  with the extracted attributes (`cat_*`, `brand`, `Product_Type`,
  `Material`, `Color`, ...) plus `title_cleaned`.
- **Comments / reviews** — `data/Home_and_Kitchen_filtered.csv`: one row per
  review (the interactions), with `reviewerID`, `asin`, `overall`,
  `unixReviewTime`, etc.

We connect them with a left join of the reviews onto the item features so
each interaction row also carries its item's extracted features.

In [21]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

# --- Item features (one row per asin) ---
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")
print("df_features:", df_features.shape)
df_features.head(3)

df_features: (1134566, 83)


,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,asin,date,imageURL,imageURLHighRes,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,title_cleaned,extracted_features_title,description_cleaned,extracted_features_description,feature_cleaned,extracted_features_feature,extracted_features,Bar_Pressure,Brand,Capacity,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,capacity_numeric,capacity_unit,capacity_volume_numeric,capacity_volume_unit,piece_count_numeric,piece_count_unit,thread_count_numeric,thread_count_unit,weight_numeric,weight_unit,bar_pressure_numeric,capacity_cups_numeric,density_weight_lb,pocket_depth_in,power_rating_w,stage_count_numeric,voltage_numeric,dimension_1,dimension_2,dimension_3,dimension_unit,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,density_weight_lb_cleaned,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,voltage_numeric_cleaned,thread_count_numeric_cleaned,weight_numeric_cleaned
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,0001487795,"October 8, 2006",[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates,special today red plate red pen,{'Color': 'red'},time honored tradition among early american fa...,{'Color': 'red'},,{},{'Color': 'red'},None,None,None,None,None,red,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"['Home & Kitchen', 'Home Dcor', 'Candles & Hol...",NaN,['VICKS INHALER relieves stuffy noses helps si...,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,NaN,Vicks,[],"['>#1,763,185 in Home & Kitchen (See Top 100 i...",Amazon Home,$4.05,0002020300,NaN,[],[],Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None,vicks inhaler relief cold sinus nasal congesti...,{},vicks inhaler relief stuffy nose help sinus co...,{},,{},{},None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,"['16 oz squeeze bottle, 1 lb.']",Artistic Churchware Communion Cup Filler: RW525,NaN,Artistic Churchware,"['Religious Supply Center', 'RW-525', 'Communi...","['>#2,127,003 in Home & Kitchen (See Top 100 i...",Amazon Home,$12.48,0006564224,NaN,[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None,artistic churchware communion cup filler rw525,{'Product_Type': 'cup'},16 oz squeeze bottle 1 lb,{'Capacity_Volume': '16 oz'},religious supply center rw-525 communion cup f...,{'Product_Type': 'cup'},"{'Product_Type': 'cup', 'Capacity_Volume': '16...",None,None,None,None,16 oz,None,None,None,None,None,None,None,None,None,None,cup,None,None,None,None,None,None,None,None,None,None,NaN,NaN,16.0,oz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# --- Comments / reviews (one row per interaction) ---
review_cols = [
    "reviewerID", "asin", "overall",
    "verified", "unixReviewTime", "reviewTime", "vote",
]
df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    usecols=review_cols,
)
print("df_reviews:", df_reviews.shape)
df_reviews.head(3)

/var/folders/86/_khp3pb10vg5vtbr6fmndrxc0000gn/T/ipykernel_31795/645899721.py:6: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv(


df_reviews: (6898955, 7)


,overall,verified,reviewTime,reviewerID,asin,unixReviewTime,vote
0,5.0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,NaN
1,3.0,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,2
2,5.0,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,NaN


In [23]:
# --- Connect the two on `asin` ---
# Attach ALL extracted feature variables plus their parsed measures.

# 1. Every extracted feature field column (the keys present in the dicts):
#    Product_Type, Material, Color, Weight, Dimensions, Brand, Theme, ...
field_cols = sorted({
    k for d in df_features["extracted_features"]
    if isinstance(d, dict) for k in d
})

# 2. Their parsed measures: numeric value + unit (+ dimension_1/2/3).
#    Use the range-cleaned numeric variant wherever one exists.
measure_cols = [
    c for c in df_features.columns
    if c.endswith("_numeric") or c.endswith("_unit")
    or c.startswith("dimension_")
    or c in ("density_weight_lb", "pocket_depth_in", "power_rating_w")
]
measure_cols = sorted({
    f"{c}_cleaned" if f"{c}_cleaned" in df_features.columns else c
    for c in measure_cols
})

# 3. Context columns to carry along (asin is the join key).
context_cols = ["asin", "cat_2", "cat_3", "cat_4", "brand", "extracted_features"]

keep_cols = list(dict.fromkeys(context_cols + field_cols + measure_cols))
keep_cols = [c for c in keep_cols if c in df_features.columns]

df = df_reviews.merge(
    df_features[keep_cols],
    on="asin",
    how="left",
    validate="many_to_one",   # many reviews -> one item row
    indicator=True,
)
n_unmatched = (df["_merge"] == "left_only").sum()
df = df.drop(columns="_merge")

print(f"merged: {df.shape}  ({len(keep_cols)} item cols attached)")
print(f"  feature fields : {len(field_cols)}")
print(f"  measure cols   : {len(measure_cols)}")
print(f"unique users: {df['reviewerID'].nunique():,} | "
      f"unique items: {df['asin'].nunique():,}")
print(f"reviews with no matching item features: {n_unmatched:,}")
df.head(5)

merged: (6898955, 59)  (53 item cols attached)
  feature fields : 26
  measure cols   : 21
unique users: 777,242 | unique items: 189,172
reviews with no matching item features: 1,134,069


,overall,verified,reviewTime,reviewerID,asin,unixReviewTime,vote,cat_2,cat_3,cat_4,brand,extracted_features,Bar_Pressure,Brand,Capacity,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,capacity_numeric,capacity_unit,capacity_volume_numeric,capacity_volume_unit,density_weight_lb_cleaned,dimension_1,dimension_2,dimension_3,dimension_unit,piece_count_numeric,piece_count_unit,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,thread_count_numeric_cleaned,thread_count_unit,voltage_numeric_cleaned,weight_numeric_cleaned,weight_unit
0,5.0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,NaN,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,2,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5.0,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,NaN,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,1383091200,NaN,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,1379635200,NaN,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**First**, I need to standardize the numerical values of features and make them in the same units (e.g., inch -> cm). Once I do that, I may need to exclude some of the feature variables.

**Variables to Exclude** <br>
**Bar_Pressure:** a bar_pressure_numeric_cleaned variable is created that includes the necessary information <br>
**extracted_features:** every features is stored as a separate variables <br>
**Capacity_Cups:** capacity_cups_numeric_cleaned is the cleaned version <br>
**Density_weight:** density_weight_lb_cleaned is the cleaned version <br>
**Pocket_Depth:** pocket_depth_in_cleaned is the cleaned version <br>
**Power_Rating:** power_rating_w_cleaned is the cleaned version <br>
**Stage_Count:** stage_count_numeric_cleaned is the cleaned version <br>
**Voltage:** voltage_numeric_cleaned is the cleaned version <br>
**Thread_Count:** thread_count_numeric_cleaned and thread_count_unit are the cleaned versions <br>
**Weight:** weight_numeric_cleaned and weight_unit are the cleaned versions <br>
**Capacity:** capacity_numeric and capacity_unit are the cleaned versions <br>
**Capacity_Volume:** capacity_volume_numeric and capacity_volume_unit are the cleaned versions <br>
**Piece_Count:** piece_count_numeric and piece_count_unit are the cleaned versions

## **Variables to Check** <br>
**Categorical Variables** <br>
brand <br>
Brand <br>
Color <br>
Features <br>
Filter_Rating <br>
Material <br>
Part_Number <br>
Product_Type <br>
Scent <br>
Shape <br>
Shape_Style <br>
Size <br>
Sub_Type <br>
Theme <br>
<br>
**Variables consisting of two** <br>
capacity_numeric <br>
capacity_unit <br>
<br>
capacity_volume_numeric <br>
capacity_volume_unit <br>
<br>
piece_count_numeric <br>
piece_count_unit <br>
<br>
thread_count_numeric <br>
thread_count_unit <br>
<br>
weight_numeric <br>
weight_unit <br>
<br>
**Variables with Range Filters** <br>
bar_pressure_numeric_cleaned <br>
capacity_cups_numeric_cleaned <br>
density_weight_lb_cleaned <br>
pocket_depth_in_cleaned <br>
power_rating_w_cleaned <br>
stage_count_numeric_cleaned <br>
voltage_numeric_cleaned <br>
<br>
**Dimension Variables** <br>
dimension_1 <br>
dimension_2 <br>
dimension_3 <br>
dimension_unit <br>
<br>
**Product Categories** <br>
cat_2 <br>
cat_3 <br>
cat_4 <br>


In [38]:
# Check categories: cat_2, cat_3, and cat_4
print(f'Unique cat_2 values: \n {df['cat_2'].unique()}')
print(f'Unique cat_3 values: \n {df['cat_3'].unique()}')
print(f'Unique cat_4 values: \n {df['cat_4'].unique()}')

Unique cat_2 values: 
 ['Home Dcor' 'Kitchen & Dining' 'Bedding' 'Wall Art' 'Furniture' nan
 'Bath' "Kids' Home Store"]
Unique cat_3 values: 
 ['Home Dcor Accents' 'Travel & To-Go Drinkware' 'Clocks'
 'Dining & Entertaining' 'Bed Pillows & Positioners' 'Posters & Prints'
 'Storage & Organization' 'Kitchen Utensils & Gadgets' "Kids' Furniture"
 "Kids' Room Dcor" nan 'Duvets, Covers & Sets' 'Kitchen & Table Linens'
 'Cutlery & Knife Accessories' "Kids' Bedding" 'Window Treatment Hardware'
 'Artificial Plants & Flowers' 'Candles & Holders' 'Cookware'
 'Bathroom Accessories' 'Small Appliances' 'Coffee, Tea & Espresso'
 'Home Brewing & Wine Making' 'Water Coolers & Filters' 'Bakeware'
 'Paintings' 'Small Appliance Parts & Accessories'
 'Photo Albums & Accessories' 'Towels' 'Living Room Furniture'
 'Home Office Furniture' 'Picture Frames' 'Home Fragrance'
 'Kitchen & Dining Room Furniture' 'Decorative Pillows, Inserts & Covers'
 'Slipcovers' 'Area Rugs, Runners & Pads'
 'Game & Recreation Ro

Check **categorical** variables

In [122]:
for i in ('brand', 'Color', 'Features', 'Filter_Rating', 'Material', 'Part_Number',
          'Product_Type', 'Scent', 'Shape', 'Shape_Style', 'Size', 'Sub_Type', 'Theme'):
    print(f'Unique number of {i}: {df[i].nunique()}')
    print(f'First 10 unique {i}: \n {df[i].unique()[:10]} \n')

Unique number of brand: 27909
First 10 unique brand: 
 ['WELLAND' 'Timolino' 'Judy Instructo' 'Communion' 'Five Star' 'Cavallini'
 'Roman' 'Bendon' 'Art By Akiane' 'Art and SoulWorks LLC'] 

Unique number of Color: 67
First 10 unique Color: 
 ['black' None 'green' 'natural' 'colorful' nan 'grey' 'blue' 'stainless'
 'orange'] 

Unique number of Features: 560
First 10 unique Features: 
 ['floating shelf' None 'decorative' 'hand painted' 'original'
 'reproduction' 'fine art' 'safe' 'magnetic' nan] 

Unique number of Filter_Rating: 105
First 10 unique Filter_Rating: 
 [None nan '5 micron' '13 gallon' '40 gallon' '99 mic' '100 gallon'
 '10 micron' '1 micron' '600 gallon'] 

Unique number of Material: 216
First 10 unique Material: 
 [None 'stainless' 'plastic' 'paper' 'vinyl' 'canvas' 'led' nan 'cotton'
 'stainless steel'] 

Unique number of Part_Number: 119
First 10 unique Part_Number: 
 [None nan 'part 102530-000-000' 'part 1' 'part number 024997-010-089'
 'part number 26093' 'part number 

Check for numeric **cleaned variables**

In [124]:
for i in ('bar_pressure_numeric_cleaned', 'capacity_cups_numeric_cleaned',
          'density_weight_lb_cleaned', 'pocket_depth_in_cleaned',
          'power_rating_w_cleaned', 'stage_count_numeric_cleaned', 'voltage_numeric_cleaned'):

    print(f'Summary of {i}: \n {df[i].describe()}')

Summary of bar_pressure_numeric_cleaned: 
 count   9,478.00
mean       15.78
std         3.49
min         2.00
25%        15.00
50%        15.00
75%        19.00
max        20.00
Name: bar_pressure_numeric_cleaned, dtype: float64
Summary of capacity_cups_numeric_cleaned: 
 count   94,958.00
mean         8.63
std          4.23
min          1.00
25%          5.00
50%          9.00
75%         12.00
max         30.00
Name: capacity_cups_numeric_cleaned, dtype: float64
Summary of density_weight_lb_cleaned: 
 count   5,194.00
mean        4.21
std         3.34
min         1.00
25%         3.00
50%         4.00
75%         4.00
max        27.60
Name: density_weight_lb_cleaned, dtype: float64
Summary of pocket_depth_in_cleaned: 
 count   30,052.00
mean        15.87
std          3.77
min          5.00
25%         14.00
50%         16.00
75%         18.00
max         25.00
Name: pocket_depth_in_cleaned, dtype: float64
Summary of power_rating_w_cleaned: 
 count   187,583.00
mean        711.37
std

In [96]:
df[df['piece_count_unit'].isna() == False][['asin', 'piece_count_numeric', 'piece_count_unit']].head(10)
# df['piece_count_unit'].unique()

,asin,piece_count_numeric,piece_count_unit
41,1605160113,10.00,pack
42,1605160113,10.00,pack
43,1605160113,10.00,pack
44,1605160113,10.00,pack
47,1605160113,10.00,pack
54,1605160113,10.00,pack
55,1605160113,10.00,pack
59,1605160113,10.00,pack
60,1605160113,10.00,pack
61,1605160113,10.00,pack


In [99]:
df[df['weight_numeric_cleaned'].isna() == False]['weight_unit'].unique()

array(['lb', 'g'], dtype=object)